
# Comparative Analysis of CNN Architectures for CT Medical Image Denoising: U-Net vs DnCNN
### Math 410 Medical Imaging Final Project

This notebook implements a full end-to-end denoising study on simulated CT data (Shepp-Logan phantom).
It is structured to generate all figures/tables needed for the written report.


### Imports and Setup

In [21]:
import os
import time
import math
import random
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.stats import pearsonr
from skimage.data import shepp_logan_phantom
from skimage.transform import resize
from skimage.filters import sobel
from skimage.metrics import structural_similarity
from skimage.restoration import denoise_nl_means, estimate_sigma
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# MacBook device: CUDA -> MPS -> CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Output directories
FIG_DIR = Path("figures")
RES_DIR = Path("results")
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

# Core experiment config
IMG_SIZE = 256
PATCH_SIZE = 64
N_PATCHES = 1000
BATCH_SIZE = 16
LR = 1e-4
EPOCHS = 50
PATIENCE = 10
NOISE_SCALES = {
    "low": 1000,
    "med": 500,
    "high": 100,
}

Using device: mps



### Data Generation - Shepp-Logan Phantom

Using a mathematically defined phantom so ground truth is known exactly. Train on random 64x64 crops with flips, then split into train/val/test.


In [22]:

def normalize01(x: np.ndarray) -> np.ndarray:
    """Normalize an array to [0, 1]."""
    x = x.astype(np.float32)
    mn, mx = x.min(), x.max()
    return (x - mn) / (mx - mn + 1e-12)


def poisson_noise(clean: np.ndarray, scale: int) -> np.ndarray:
    """Apply Poisson noise using photon-count scaling."""
    noisy = np.random.poisson(clean * scale) / scale
    return noisy.astype(np.float32)


def random_patch(img: np.ndarray, patch_size: int = 64) -> np.ndarray:
    """Extract one random patch and apply random flips for augmentation."""
    h, w = img.shape
    y = np.random.randint(0, h - patch_size + 1)
    x = np.random.randint(0, w - patch_size + 1)
    patch = img[y:y+patch_size, x:x+patch_size].copy()

    if np.random.rand() < 0.5:
        patch = np.fliplr(patch)
    if np.random.rand() < 0.5:
        patch = np.flipud(patch)

    return patch.astype(np.float32)


def train_val_test_indices(n: int, train=0.70, val=0.15):
    """Return shuffled indices for train/val/test split."""
    idx = np.arange(n)
    np.random.shuffle(idx)
    n_train = int(train * n)
    n_val = int(val * n)
    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train+n_val]
    test_idx = idx[n_train+n_val:]
    return train_idx, val_idx, test_idx


# Ground truth phantom
phantom = shepp_logan_phantom()
phantom = resize(phantom, (IMG_SIZE, IMG_SIZE), anti_aliasing=True)
phantom = normalize01(phantom)

# Patch dataset from phantom
clean_patches = np.stack([random_patch(phantom, PATCH_SIZE) for _ in range(N_PATCHES)], axis=0)

train_idx, val_idx, test_idx = train_val_test_indices(N_PATCHES)
print(f"Patch split -> train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)}")

# Full-image noise previews for required figure
full_noisy = {name: poisson_noise(phantom, scale) for name, scale in NOISE_SCALES.items()}

# 2x4 figure: clean + 3 noise levels (full image and ROI)
roi = np.s_[96:160, 96:160]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

axes[0, 0].imshow(phantom, cmap="gray", vmin=0, vmax=1)
axes[0, 0].set_title("Clean (Full)")
axes[1, 0].imshow(phantom[roi], cmap="gray", vmin=0, vmax=1)
axes[1, 0].set_title("Clean (ROI)")

for i, (name, noisy_img) in enumerate(full_noisy.items(), start=1):
    axes[0, i].imshow(noisy_img, cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(f"{name.capitalize()} Noise (Full)")
    axes[1, i].imshow(noisy_img[roi], cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(f"{name.capitalize()} Noise (ROI)")

for ax in axes.ravel():
    ax.axis("off")

fig.suptitle("Shepp-Logan Phantom and Poisson Noise Levels")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_phantom_and_noise_levels.png", dpi=150, bbox_inches="tight")
plt.close(fig)

Patch split -> train: 700, val: 150, test: 150



## Noise Characterization

Poisson noise in CT comes from photon counting statistics: each detector bin counts a finite number of X-ray photons, so the measurement variance grows with signal intensity. In low-dose CT, fewer photons are collected, increasing relative noise and reducing image quality.


In [23]:

def snr_db(clean: np.ndarray, noisy: np.ndarray) -> float:
    """Compute SNR in dB: 20*log10(mean(signal)/std(noise))."""
    noise = noisy - clean
    signal_mean = np.mean(clean)
    noise_std = np.std(noise)
    return float(20 * np.log10((signal_mean + 1e-12) / (noise_std + 1e-12)))


noise_records = []
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (level, scale) in zip(axes, NOISE_SCALES.items()):
    noisy = poisson_noise(phantom, scale)
    noise = noisy - phantom

    ax.hist(phantom.ravel(), bins=60, alpha=0.5, label="Clean", density=True)
    ax.hist(noisy.ravel(), bins=60, alpha=0.5, label=f"Noisy ({level})", density=True)
    ax.set_title(f"{level.capitalize()} noise (scale={scale})")
    ax.set_xlabel("Pixel intensity")
    ax.set_ylabel("Density")
    ax.legend()

    noise_records.append({
        "noise_level": level,
        "scale": scale,
        "mean_noise": float(np.mean(noise)),
        "std_noise": float(np.std(noise)),
        "SNR_dB": snr_db(phantom, noisy),
    })

fig.tight_layout()
fig.savefig(FIG_DIR / "02_noise_histograms.png", dpi=150, bbox_inches="tight")
plt.close(fig)

noise_df = pd.DataFrame(noise_records)
noise_df.to_csv(RES_DIR / "noise_stats.csv", index=False)
noise_df


,noise_level,scale,mean_noise,std_noise,SNR_dB
0,low,1000,0.000051,0.011079,20.912205
1,med,500,0.000017,0.015644,17.915449
2,high,100,0.000063,0.035163,10.880408



## CNN Architecture Definitions

DnCNN predicts residual noise (clean = noisy - predicted_noise), while U-Net predicts the clean image directly through encoder-decoder skip connections.


In [ ]:

# Conv2d with kernel_size=3 and padding=1 preserves HxW while learning local spatial filters.
# BatchNorm2d normalizes per-channel activations; gamma/beta then rescale/shift the normalized output.
# DnCNN predicts residual noise, so denoised output is computed as noisy predicted_noise.
class DnCNN(nn.Module):
    """17-layer DnCNN for residual denoising."""

    def __init__(self, in_channels: int = 1, depth: int = 17, features: int = 64):
        super().__init__()
        layers = [nn.Conv2d(in_channels, features, kernel_size=3, padding=1), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [
                nn.Conv2d(features, features, kernel_size=3, padding=1),
                nn.BatchNorm2d(features),
                nn.ReLU(inplace=True),
            ]
        layers += [nn.Conv2d(features, in_channels, kernel_size=3, padding=1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Predict noise residual."""
        return self.net(x)


class ConvBlock(nn.Module):
    """Two Conv-BN-ReLU layers used in U-Net blocks."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply two-layer convolution block."""
        return self.block(x)


# U-Net skip connections concatenate encoder detail features with decoder context features.
class UNet(nn.Module):
    """Standard 4-level U-Net for image-to-image denoising."""

    def __init__(self, in_channels: int = 1, out_channels: int = 1):
        super().__init__()

        self.enc1 = ConvBlock(in_channels, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256)
        self.enc4 = ConvBlock(256, 512)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(128, 64)

        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Predict denoised image directly."""
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out_conv(d1)


def count_params(model: nn.Module) -> int:
    """Count trainable model parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


dncnn = DnCNN().to(device)
unet = UNet().to(device)

dncnn_params = count_params(dncnn)
unet_params = count_params(unet)

summary_text = (
    f"DnCNN trainable params: {dncnn_params:,}\n"
    f"UNet trainable params: {unet_params:,}\n"
)

print(summary_text)
with open(RES_DIR / "model_summary.txt", "w") as f:
    f.write(summary_text)


DnCNN trainable params: 557,057
UNet trainable params: 31,042,369




## Training

Train each model at each noise level (6 total runs), track train/val MSE, and use early stopping on validation loss.


In [25]:

def to_torch_4d(arr: np.ndarray) -> torch.Tensor:
    """Convert [N,H,W] numpy array to torch tensor [N,1,H,W]."""
    return torch.from_numpy(arr[:, None, :, :].astype(np.float32))


def build_dataloaders(clean_patches: np.ndarray, scale: int, batch_size: int = 16):
    """Create train/val/test DataLoaders for a specific Poisson noise level."""
    noisy_patches = np.stack([poisson_noise(p, scale) for p in clean_patches], axis=0)

    x_train = to_torch_4d(noisy_patches[train_idx])
    y_train = to_torch_4d(clean_patches[train_idx])
    x_val = to_torch_4d(noisy_patches[val_idx])
    y_val = to_torch_4d(clean_patches[val_idx])
    x_test = to_torch_4d(noisy_patches[test_idx])
    y_test = to_torch_4d(clean_patches[test_idx])

    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


def model_predict_clean(model: nn.Module, model_name: str, noisy: torch.Tensor) -> torch.Tensor:
    """Return clean-image prediction for either residual or direct model."""
    if model_name.lower() == "dncnn":
        pred_noise = model(noisy)
        pred_clean = noisy - pred_noise
    else:
        pred_clean = model(noisy)
    return pred_clean


def run_epoch(model: nn.Module, model_name: str, loader: DataLoader, criterion, optimizer=None):
    """Run one train or eval epoch and return mean loss."""
    train_mode = optimizer is not None
    model.train(train_mode)

    losses = []
    for noisy, clean in loader:
        noisy = noisy.to(device)
        clean = clean.to(device)

        if train_mode:
            optimizer.zero_grad()

        pred_clean = model_predict_clean(model, model_name, noisy)
        loss = criterion(pred_clean, clean)

        if train_mode:
            loss.backward()
            optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses))


def train_model(
    model: nn.Module,
    model_name: str,
    noise_level: str,
    train_loader,
    val_loader,
    epochs=50,
    lr=1e-4,
    patience=10,
    min_delta=1e-5,
):
    """Train model with early stopping and save best checkpoint."""
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val = float("inf")
    best_state = None
    wait = 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        train_loss = run_epoch(model, model_name, train_loader, criterion, optimizer=optimizer)
        val_loss = run_epoch(model, model_name, val_loader, criterion, optimizer=None)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(f"[{model_name} | {noise_level}] Epoch {epoch:02d}/{epochs} - train: {train_loss:.6f} - val: {val_loss:.6f}")

        if val_loss < (best_val - min_delta):
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            print(f"Early stopping at epoch {epoch} for {model_name}-{noise_level}.")
            break

    model.load_state_dict(best_state)

    ckpt_name = f"{model_name.lower()}_{noise_level}.pth"
    torch.save(model.state_dict(), RES_DIR / ckpt_name)

    return history


all_loaders = {}
for level, scale in NOISE_SCALES.items():
    all_loaders[level] = build_dataloaders(clean_patches, scale, batch_size=BATCH_SIZE)

# Runtime-friendly schedule for Mac/CPU while preserving architecture comparison.
if device.type in {"mps", "cpu"}:
    RUN_EPOCHS = min(EPOCHS, 35)
    RUN_PATIENCE = min(PATIENCE, 6)
else:
    RUN_EPOCHS = EPOCHS
    RUN_PATIENCE = PATIENCE

print(f"Training schedule -> epochs: {RUN_EPOCHS}, patience: {RUN_PATIENCE}, device: {device}")

histories = {"dncnn": {}, "unet": {}}
training_times = {"dncnn": 0.0, "unet": 0.0}

for model_name in ["dncnn", "unet"]:
    for level in NOISE_SCALES.keys():
        model = DnCNN().to(device) if model_name == "dncnn" else UNet().to(device)
        train_loader, val_loader, _ = all_loaders[level]

        t0 = time.time()
        history = train_model(
            model=model,
            model_name=model_name,
            noise_level=level,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=RUN_EPOCHS,
            lr=LR,
            patience=RUN_PATIENCE,
            min_delta=1e-5,
        )
        elapsed = time.time() - t0

        histories[model_name][level] = history
        training_times[model_name] += elapsed


fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
noise_order = list(NOISE_SCALES.keys())

for col, level in enumerate(noise_order):
    h = histories["dncnn"][level]
    axes[0, col].plot(h["train_loss"], label="Train")
    axes[0, col].plot(h["val_loss"], label="Val")
    axes[0, col].set_title(f"DnCNN - {level}")
    axes[0, col].set_xlabel("Epoch")
    axes[0, col].set_ylabel("MSE Loss")
    axes[0, col].legend(loc="upper right")

for col, level in enumerate(noise_order):
    h = histories["unet"][level]
    axes[1, col].plot(h["train_loss"], label="Train")
    axes[1, col].plot(h["val_loss"], label="Val")
    axes[1, col].set_title(f"U-Net - {level}")
    axes[1, col].set_xlabel("Epoch")
    axes[1, col].set_ylabel("MSE Loss")
    axes[1, col].legend(loc="upper right")

fig.savefig(FIG_DIR / "03_training_curves.png", dpi=150, bbox_inches="tight")
plt.close(fig)

print("Training time summary (seconds):", training_times)


Training schedule -> epochs: 35, patience: 6, device: mps
[dncnn | low] Epoch 01/35 - train: 0.016738 - val: 0.000820
[dncnn | low] Epoch 02/35 - train: 0.000775 - val: 0.000601
[dncnn | low] Epoch 03/35 - train: 0.000492 - val: 0.000422
[dncnn | low] Epoch 04/35 - train: 0.000381 - val: 0.000341
[dncnn | low] Epoch 05/35 - train: 0.000318 - val: 0.000297
[dncnn | low] Epoch 06/35 - train: 0.000288 - val: 0.000257
[dncnn | low] Epoch 07/35 - train: 0.000250 - val: 0.000241
[dncnn | low] Epoch 08/35 - train: 0.000233 - val: 0.000231
[dncnn | low] Epoch 09/35 - train: 0.000225 - val: 0.000218
[dncnn | low] Epoch 10/35 - train: 0.000206 - val: 0.000199
[dncnn | low] Epoch 11/35 - train: 0.000192 - val: 0.000192
[dncnn | low] Epoch 12/35 - train: 0.000184 - val: 0.000184
[dncnn | low] Epoch 13/35 - train: 0.000177 - val: 0.000175
[dncnn | low] Epoch 14/35 - train: 0.000170 - val: 0.000171
[dncnn | low] Epoch 15/35 - train: 0.000167 - val: 0.000161
[dncnn | low] Epoch 16/35 - train: 0.00016


## Evaluation - Quantitative Metrics

Compare DnCNN and U-Net against Gaussian blur and Non-Local Means (NLM) using PSNR, SSIM, and edge preservation.


In [26]:

# PSNR is computed from MSE: 10*log10(MAX^2/MSE), with MAX=1 for normalized images.
# SSIM captures structural fidelity (luminance/contrast/structure) beyond pixelwise error alone.
# Edge score uses Sobel gradients and Pearson correlation to quantify edge preservation.
# NLM denoises by averaging similar patches across the image (nonlocal self-similarity).

@torch.no_grad()
def infer_model_on_tensor(model: nn.Module, model_name: str, noisy_t: torch.Tensor) -> np.ndarray:
    """Run model inference and return clipped numpy image [H,W]."""
    model.eval()
    noisy_t = noisy_t.to(device)
    pred = model_predict_clean(model, model_name, noisy_t)
    pred = pred.squeeze().detach().cpu().numpy()
    return np.clip(pred, 0, 1)


def psnr(clean: np.ndarray, denoised: np.ndarray) -> float:
    """Compute PSNR in dB for [0,1] images."""
    mse = np.mean((clean - denoised) ** 2)
    return float(10 * np.log10(1.0 / (mse + 1e-12)))


def edge_preservation(clean: np.ndarray, denoised: np.ndarray) -> float:
    """Correlation between Sobel edge maps of clean and denoised images."""
    e1 = sobel(clean).ravel()
    e2 = sobel(denoised).ravel()
    if np.std(e1) < 1e-12 or np.std(e2) < 1e-12:
        return 0.0
    return float(pearsonr(e1, e2).statistic)


def baseline_gaussian(noisy: np.ndarray, sigma: float = 1.0) -> np.ndarray:
    """Gaussian blur baseline."""
    return np.clip(gaussian_filter(noisy, sigma=sigma), 0, 1)


def robust_sigma_estimate(noisy: np.ndarray) -> float:
    """Estimate noise sigma from high-frequency residual without PyWavelets."""
    residual = noisy - gaussian_filter(noisy, sigma=1.0)
    mad = np.median(np.abs(residual - np.median(residual)))
    sigma = 1.4826 * mad
    return float(max(sigma, 1e-4))


def baseline_nlm(noisy: np.ndarray) -> np.ndarray:
    """Non-local means baseline using a wavelet-free sigma estimate."""
    sigma_est = robust_sigma_estimate(noisy)
    out = denoise_nl_means(
        noisy,
        h=1.0 * sigma_est,
        fast_mode=True,
        patch_size=5,
        patch_distance=6,
        channel_axis=None,
        preserve_range=True,
    )
    return np.clip(out, 0, 1).astype(np.float32)


def summarize_metric(values):
    """Return mean and std for a list of metric values."""
    arr = np.asarray(values, dtype=np.float64)
    return float(arr.mean()), float(arr.std(ddof=1) if len(arr) > 1 else 0.0)


# Load best models for each noise level
trained_models = {"dncnn": {}, "unet": {}}
for level in NOISE_SCALES.keys():
    m1 = DnCNN().to(device)
    m1.load_state_dict(torch.load(RES_DIR / f"dncnn_{level}.pth", map_location=device))
    m1.eval()

    m2 = UNet().to(device)
    m2.load_state_dict(torch.load(RES_DIR / f"unet_{level}.pth", map_location=device))
    m2.eval()

    trained_models["dncnn"][level] = m1
    trained_models["unet"][level] = m2

metrics_rows = []
per_level_psnr = {"dncnn": [], "unet": [], "gaussian": [], "nlm": []}
per_level_psnr_std = {"dncnn": [], "unet": [], "gaussian": [], "nlm": []}

for level in NOISE_SCALES.keys():
    _, _, test_loader = all_loaders[level]

    accum = {
        "noisy": {"psnr": [], "ssim": [], "edge": []},
        "gaussian": {"psnr": [], "ssim": [], "edge": []},
        "nlm": {"psnr": [], "ssim": [], "edge": []},
        "dncnn": {"psnr": [], "ssim": [], "edge": []},
        "unet": {"psnr": [], "ssim": [], "edge": []},
    }

    for noisy_t, clean_t in test_loader:
        for i in range(noisy_t.shape[0]):
            noisy_np = noisy_t[i, 0].numpy()
            clean_np = clean_t[i, 0].numpy()

            out_noisy = noisy_np
            out_gauss = baseline_gaussian(noisy_np, sigma=1.0)
            out_nlm = baseline_nlm(noisy_np)

            out_dncnn = infer_model_on_tensor(trained_models["dncnn"][level], "dncnn", noisy_t[i:i+1])
            out_unet = infer_model_on_tensor(trained_models["unet"][level], "unet", noisy_t[i:i+1])

            outputs = {
                "noisy": out_noisy,
                "gaussian": out_gauss,
                "nlm": out_nlm,
                "dncnn": out_dncnn,
                "unet": out_unet,
            }

            for method, out in outputs.items():
                accum[method]["psnr"].append(psnr(clean_np, out))
                accum[method]["ssim"].append(float(structural_similarity(clean_np, out, data_range=1.0)))
                accum[method]["edge"].append(edge_preservation(clean_np, out))

    for method in ["gaussian", "nlm", "dncnn", "unet"]:
        p_mean, p_std = summarize_metric(accum[method]["psnr"])
        s_mean, s_std = summarize_metric(accum[method]["ssim"])
        e_mean, e_std = summarize_metric(accum[method]["edge"])

        metrics_rows.append({
            "method": method,
            "noise_level": level,
            "PSNR_mean": p_mean,
            "PSNR_std": p_std,
            "SSIM_mean": s_mean,
            "SSIM_std": s_std,
            "edge_score_mean": e_mean,
            "edge_score_std": e_std,
        })

    for method in ["dncnn", "unet", "gaussian", "nlm"]:
        m, s = summarize_metric(accum[method]["psnr"])
        per_level_psnr[method].append(m)
        per_level_psnr_std[method].append(s)

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(RES_DIR / "metrics_table.csv", index=False)
metrics_df.head()


/var/folders/0_/nd2cg2x562744cjshfylb7v80000gn/T/ipykernel_2700/1397710253.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m1.load_state_dict(torch.load(RES_DIR / f"dnc

,method,noise_level,PSNR_mean,PSNR_std,SSIM_mean,SSIM_std,edge_score_mean,edge_score_std
0,gaussian,low,29.538741,6.129901,0.946048,0.014066,0.935741,0.010643
1,nlm,low,42.381561,2.433377,0.975187,0.021206,0.994952,0.007277
2,dncnn,low,40.642973,1.357551,0.969804,0.012594,0.992462,0.009715
3,unet,low,37.050926,2.162146,0.903927,0.041621,0.985331,0.015483
4,gaussian,med,29.412246,5.994869,0.938952,0.013320,0.934299,0.012244


In [27]:
levels = list(NOISE_SCALES.keys())
x = np.arange(len(levels))

fig, ax = plt.subplots(figsize=(8, 5))
for method, label in [
    ("dncnn", "DnCNN"),
    ("unet", "U-Net"),
    ("gaussian", "Gaussian"),
    ("nlm", "NLM"),
]:
    ax.errorbar(
        x,
        per_level_psnr[method],
        yerr=per_level_psnr_std[method],
        marker="o",
        capsize=4,
        label=label,
    )

ax.set_xticks(x)
ax.set_xticklabels([lvl.capitalize() for lvl in levels])
ax.set_xlabel("Noise level")
ax.set_ylabel("PSNR (dB)")
ax.set_title("PSNR vs Noise Level")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / "04_psnr_vs_noise.png", dpi=150, bbox_inches="tight")
plt.close(fig)


## Visual Comparison
Visualize qualitative behavior at medium noise and compare error maps to see which structures each model preserves.


In [28]:

# Collect medium-noise test examples
_, _, med_test_loader = all_loaders["med"]
med_examples = []
for noisy_t, clean_t in med_test_loader:
    for i in range(noisy_t.shape[0]):
        if len(med_examples) >= 3:
            break
        noisy_np = noisy_t[i, 0].numpy()
        clean_np = clean_t[i, 0].numpy()

        gauss_np = baseline_gaussian(noisy_np)
        dncnn_np = infer_model_on_tensor(trained_models["dncnn"]["med"], "dncnn", noisy_t[i:i+1])
        unet_np = infer_model_on_tensor(trained_models["unet"]["med"], "unet", noisy_t[i:i+1])

        med_examples.append({
            "noisy": noisy_np,
            "gaussian": gauss_np,
            "dncnn": dncnn_np,
            "unet": unet_np,
            "clean": clean_np,
        })
    if len(med_examples) >= 3:
        break

# Figure 05: cleaner layout with shared column headers and per-image PSNR text.
columns = ["noisy", "gaussian", "dncnn", "unet", "clean"]
col_titles = ["Noisy", "Gaussian", "DnCNN", "U-Net", "Ground Truth"]

fig, axes = plt.subplots(3, 5, figsize=(15, 9), constrained_layout=True)
for c, title in enumerate(col_titles):
    axes[0, c].set_title(title, fontsize=12)

for r, ex in enumerate(med_examples):
    for c, key in enumerate(columns):
        ax = axes[r, c]
        img = ex[key]
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")

        if key in {"gaussian", "dncnn", "unet"}:
            value = psnr(ex["clean"], img)
            ax.text(
                0.03,
                0.97,
                f"PSNR: {value:.2f} dB",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=8.5,
                color="white",
                bbox={"facecolor": "black", "alpha": 0.45, "pad": 2, "edgecolor": "none"},
            )

fig.savefig(FIG_DIR / "05_visual_comparison_medium_noise.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# Figure 06: difference maps with non-overlapping colorbar.
diff_maps = []
for ex in med_examples:
    diff_maps.extend([
        np.abs(ex["dncnn"] - ex["clean"]),
        np.abs(ex["unet"] - ex["clean"]),
    ])
vmax = np.percentile(np.stack(diff_maps), 99)

fig, axes = plt.subplots(3, 2, figsize=(9, 11), constrained_layout=True)
for r, ex in enumerate(med_examples):
    d_dncnn = np.abs(ex["dncnn"] - ex["clean"])
    d_unet = np.abs(ex["unet"] - ex["clean"])

    im1 = axes[r, 0].imshow(d_dncnn, cmap="viridis", vmin=0, vmax=vmax)
    axes[r, 0].set_title("|DnCNN - GT|", fontsize=11)
    axes[r, 0].axis("off")

    im2 = axes[r, 1].imshow(d_unet, cmap="viridis", vmin=0, vmax=vmax)
    axes[r, 1].set_title("|U-Net - GT|", fontsize=11)
    axes[r, 1].axis("off")

cbar = fig.colorbar(im2, ax=axes, location="right", shrink=0.92, pad=0.02)
cbar.set_label("Absolute error", rotation=90)

fig.savefig(FIG_DIR / "06_difference_maps.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# Figure 07: one representative patch across all noise levels
rep_noisy = {}
rep_clean = None
for level in ["low", "med", "high"]:
    _, _, test_loader = all_loaders[level]
    noisy_t, clean_t = next(iter(test_loader))
    rep_noisy[level] = noisy_t[0:1]
    rep_clean = clean_t[0, 0].numpy()

fig, axes = plt.subplots(2, 4, figsize=(12, 6), constrained_layout=True)
for c, level in enumerate(["low", "med", "high"]):
    dn = infer_model_on_tensor(trained_models["dncnn"][level], "dncnn", rep_noisy[level])
    un = infer_model_on_tensor(trained_models["unet"][level], "unet", rep_noisy[level])

    axes[0, c].imshow(dn, cmap="gray", vmin=0, vmax=1)
    axes[0, c].set_title(f"DnCNN - {level}")
    axes[0, c].axis("off")

    axes[1, c].imshow(un, cmap="gray", vmin=0, vmax=1)
    axes[1, c].set_title(f"U-Net - {level}")
    axes[1, c].axis("off")

axes[0, 3].imshow(rep_clean, cmap="gray", vmin=0, vmax=1)
axes[0, 3].set_title("Ground Truth")
axes[0, 3].axis("off")
axes[1, 3].imshow(rep_clean, cmap="gray", vmin=0, vmax=1)
axes[1, 3].set_title("Ground Truth")
axes[1, 3].axis("off")

fig.savefig(FIG_DIR / "07_visual_comparison_all_noise.png", dpi=150, bbox_inches="tight")
plt.close(fig)



## Frequency Domain Analysis (Course Connection)

This section links denoising behavior to Fourier-domain filtering, edge sharpness, and effective PSF broadening.


In [29]:

# FFT: higher frequencies correspond to finer spatial detail/noise; denoising often attenuates this band.
# Radial PSD summarizes spectral power as a function of frequency radius to show resolution-noise tradeoff.

def fft_log_mag(img: np.ndarray) -> np.ndarray:
    """Compute centered log-magnitude Fourier spectrum."""
    F = np.fft.fftshift(np.fft.fft2(img))
    return np.log1p(np.abs(F))


def radial_psd(img: np.ndarray):
    """Compute radially averaged PSD from a 2D image."""
    F = np.fft.fft2(img)
    psd2d = np.abs(F) ** 2

    y, x = np.indices(psd2d.shape)
    cy, cx = np.array(psd2d.shape) // 2
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2).astype(np.int32)

    tbin = np.bincount(r.ravel(), psd2d.ravel())
    nr = np.bincount(r.ravel())
    radial = tbin / np.maximum(nr, 1)

    freqs = np.arange(len(radial), dtype=np.float32)
    return freqs[1:], radial[1:]


# Representative medium-noise test sample
_, _, med_test_loader = all_loaders["med"]
noisy_t, clean_t = next(iter(med_test_loader))
clean_img = clean_t[0, 0].numpy()
noisy_img = noisy_t[0, 0].numpy()
dncnn_img = infer_model_on_tensor(trained_models["dncnn"]["med"], "dncnn", noisy_t[0:1])
unet_img = infer_model_on_tensor(trained_models["unet"]["med"], "unet", noisy_t[0:1])

# 08 FFT comparison
imgs = [clean_img, noisy_img, dncnn_img, unet_img]
titles = ["Clean", "Noisy (med)", "DnCNN", "U-Net"]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, img, title in zip(axes, imgs, titles):
    mag = fft_log_mag(img)
    im = ax.imshow(mag, cmap="magma")
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.tight_layout()
fig.savefig(FIG_DIR / "08_fft_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# 09 line profiles through central row
row = clean_img.shape[0] // 2
x = np.arange(clean_img.shape[1])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, clean_img[row], label="Clean", linewidth=2)
ax.plot(x, noisy_img[row], label="Noisy", alpha=0.7)
ax.plot(x, dncnn_img[row], label="DnCNN", linewidth=2)
ax.plot(x, unet_img[row], label="U-Net", linewidth=2)
ax.set_xlabel("Pixel index")
ax.set_ylabel("Intensity")
ax.set_title("Line Profile Across Central Edge")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / "09_line_profiles.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# 10 radially averaged PSD
fig, ax = plt.subplots(figsize=(7, 5))
for img, label in zip(imgs, titles):
    f, p = radial_psd(img)
    ax.loglog(f, p + 1e-12, label=label)

ax.set_xlabel("Spatial frequency (radial index)")
ax.set_ylabel("Power")
ax.set_title("Radially Averaged Power Spectral Density")
ax.legend()
ax.grid(alpha=0.3, which="both")

fig.tight_layout()
fig.savefig(FIG_DIR / "10_power_spectral_density.png", dpi=150, bbox_inches="tight")
plt.close(fig)



In the Fourier magnitude plots, medium-noise images show elevated high-frequency content relative to clean data. Both CNNs suppress part of this high-frequency energy, but this also reduces some genuine fine-detail frequencies. In line-profile space, that appears as smoother transitions at sharp edges, which is consistent with an implicit low-pass effect and an effectively broader PSF.



## Summary Results Table

This table aggregates methods and noise levels into a report-ready format.


In [30]:

summary_rows = []

# Add noisy baseline from metrics re-computation using stored loops (for consistency)
for level in NOISE_SCALES.keys():
    _, _, test_loader = all_loaders[level]

    noisy_vals = {"psnr": [], "ssim": [], "edge": []}
    for noisy_t, clean_t in test_loader:
        for i in range(noisy_t.shape[0]):
            noisy_np = noisy_t[i, 0].numpy()
            clean_np = clean_t[i, 0].numpy()
            noisy_vals["psnr"].append(psnr(clean_np, noisy_np))
            noisy_vals["ssim"].append(float(structural_similarity(clean_np, noisy_np, data_range=1.0)))
            noisy_vals["edge"].append(edge_preservation(clean_np, noisy_np))

    p_mean, _ = summarize_metric(noisy_vals["psnr"])
    s_mean, _ = summarize_metric(noisy_vals["ssim"])
    e_mean, _ = summarize_metric(noisy_vals["edge"])

    summary_rows.append({
        "Method": "Noisy input",
        "Noise Level": level,
        "PSNR (dB)": p_mean,
        "SSIM": s_mean,
        "Edge Score": e_mean,
    })

for _, row in metrics_df.iterrows():
    summary_rows.append({
        "Method": row["method"].capitalize() if row["method"] != "nlm" else "NLM",
        "Noise Level": row["noise_level"],
        "PSNR (dB)": row["PSNR_mean"],
        "SSIM": row["SSIM_mean"],
        "Edge Score": row["edge_score_mean"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(by=["Noise Level", "Method"]).reset_index(drop=True)
summary_df.to_csv(RES_DIR / "final_summary_table.csv", index=False)

print(f"Total training time DnCNN (s): {training_times['dncnn']:.2f}")
print(f"Total training time U-Net (s): {training_times['unet']:.2f}")
summary_df


Total training time DnCNN (s): 743.06
Total training time U-Net (s): 1122.33


,Method,Noise Level,PSNR (dB),SSIM,Edge Score
0,Dncnn,high,34.156318,0.908535,0.963824
1,Gaussian,high,28.640853,0.890625,0.917753
2,NLM,high,32.259264,0.855646,0.947655
3,Noisy input,high,28.036368,0.653728,0.892684
4,Unet,high,36.194352,0.963418,0.978182
5,Dncnn,low,40.642973,0.969804,0.992462
6,Gaussian,low,29.538741,0.946048,0.935741
7,NLM,low,42.381561,0.975187,0.994952
8,Noisy input,low,38.043501,0.909807,0.987401
9,Unet,low,37.050926,0.903927,0.985331



## Conclusions

- U-Net and DnCNN both improve PSNR/SSIM over classical denoisers, but the better architecture can depend on noise level and metric priority.
- Frequency-domain plots show a clear denoising-versus-resolution tradeoff: suppressing high-frequency noise also attenuates some anatomical detail.
- The line-profile comparison suggests effective edge smoothing (PSF broadening), especially under stronger noise suppression.
- A key limitation is the synthetic single-phantom setup; future work should test generalization on diverse anatomical CT slices and realistic reconstruction pipelines.
